# Analzye collected data

Here, we gather the data collected from the different models and compare them to human data.

### Import

In [1]:
import pandas as pd
import plotly.express as px
import os
from pathlib import Path
import  matplotlib.pyplot as plt

### Functions

In [ ]:
def read_scores_from_model(model):

    model_name = model + "_"
    files = list(Path("results").glob(f"{model_name}*.csv"))

    df = pd.DataFrame([])

    print(f"Gathered data for model {model} from the following files.")
    for f in files:
        print(f.name)
        df_data = pd.read_csv(f'results/{f.name}')
        df_data["run"] = f.name[-5:-4]
        df = pd.concat([df, df_data], ignore_index=True)

    df["model"] = model
    df["id"] = df["id"].astype(int)

    return df

def summarize_scores_from_model(df, model):
    summary = df.groupby("id").agg({"score": ["mean", "std", "sem"]}).reset_index()
    summary.columns = summary.columns.get_level_values(1)
    #summary.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
    summary.columns = ['id', 'score_mean', 'score_std', 'score_sem']
    print(f"Summarized data for model {model}.")

    return summary


def collate_human_and_llm_data_wide(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        summary_model.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
        df = pd.merge(df, summary_model, left_on="CODE", right_on="id", how="left").drop(columns='id')
        

    return df


def collate_human_and_llm_data_long(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")
    df.drop(columns=["CONTEXT QUESTION", "CRITICAL UTTERANCE", "CORRECT RESPONSE (0=no; 1=yes)", "FUN_std", "FUN_sem", "COH_std", "COH_sem", "DIR_std", "DIR_sem", "PRE_std", "PRE_sem", "SSI_std", "SSI_sem", "CER_std", "CER_sem"], inplace=True) #this is not really necessary

    # add information such as condition and set
    ref = df

    # put them on the same scale as the LLm scores (0-1)
    df["FUN_mean"] = rescale(df["FUN_mean"], 1, 7)
    df["COH_mean"] = rescale(df["COH_mean"], 1, 7)
    df["DIR_mean"] = rescale(df["DIR_mean"], 1, 7)
    df["PRE_mean"] = rescale(df["PRE_mean"], 1, 7)
    df["SSI_mean"] = rescale(df["SSI_mean"], 1, 7)
    df["CER_mean"] = rescale(df["CER_mean"], 1, 4)
    df["evaluator"] = "human"

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        
        # only take mean, leave sd and std out
        summary_model  = summary_model[["id", "score_mean"]] 
        summary_model.rename(columns={"score_mean": "FUN_mean", "id": "CODE"}, inplace=True)
        
        # generate certainty score
        summary_model["CER_mean"] = (summary_model["FUN_mean"] - 0.5).abs() * 2
        
        # add evaluator column to indicate which model the scores are from
        summary_model["evaluator"] = model

        # add SET and CONDITION infformation from the reference human data
        summary_model = summary_model.merge(ref[["CODE", "SET (1=SA-matched;2=non-SA-matched)", "CONDITION"]], on="CODE", how="left", validate="m:1")

        # concatenate with data so far
        df = pd.concat([df, summary_model], axis=0, join='outer').reset_index(drop=True)
        
    # rename SET column
    df = df.rename(columns={"SET (1=SA-matched;2=non-SA-matched)": "SET"})

    return df

def rescale(series: pd.Series, min: int, max: int) -> pd.Series:
    return (series - min) / (max - min)

### Aggregation

In [55]:
MODELS = [
    "gpt-5.4",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.4-mini-2026-03-17",
]

df_wide =  collate_human_and_llm_data_wide(MODELS)
df_long = collate_human_and_llm_data_long(MODELS)


Gathered data for model gpt-5.4 from the following files.
gpt-5.4_score_run1.csv
gpt-5.4_score_run2.csv
gpt-5.4_score_run3.csv
Summarized data for model gpt-5.4.
Gathered data for model gpt-5.4-nano-2026-03-17 from the following files.
gpt-5.4-nano-2026-03-17_score_run2.csv
gpt-5.4-nano-2026-03-17_score_run3.csv
gpt-5.4-nano-2026-03-17_score_run1.csv
Summarized data for model gpt-5.4-nano-2026-03-17.
Gathered data for model gpt-5.4-mini-2026-03-17 from the following files.
gpt-5.4-mini-2026-03-17_score_run2.csv
gpt-5.4-mini-2026-03-17_score_run3.csv
gpt-5.4-mini-2026-03-17_score_run1.csv
Summarized data for model gpt-5.4-mini-2026-03-17.
Gathered data for model gpt-5.4 from the following files.
gpt-5.4_score_run1.csv
gpt-5.4_score_run2.csv
gpt-5.4_score_run3.csv
Summarized data for model gpt-5.4.
Columns in summary_model before: Index(['CODE', 'FUN_mean', 'CER_mean', 'evaluator'], dtype='str')
Ubniquevalues of SET: [1 2]
Columns in summary_model after: Index(['CODE', 'FUN_mean', 'CER_m

In [56]:
df_long.tail(20)
#df_long["SET (1=SA-matched;2=non-SA-matched)"].unique()

,SET,CODE,CONDITION,FUN_mean,COH_mean,DIR_mean,PRE_mean,SSI_mean,CER_mean,evaluator
1084,2,2255,indirect,0.663333,NaN,NaN,NaN,NaN,0.326667,gpt-5.4-mini-2026-03-17
1085,2,2256,indirect,0.916667,NaN,NaN,NaN,NaN,0.833333,gpt-5.4-mini-2026-03-17
1086,2,2257,indirect,0.396667,NaN,NaN,NaN,NaN,0.206667,gpt-5.4-mini-2026-03-17
1087,2,2259,indirect,0.916667,NaN,NaN,NaN,NaN,0.833333,gpt-5.4-mini-2026-03-17
1088,2,2260,indirect,0.916667,NaN,NaN,NaN,NaN,0.833333,gpt-5.4-mini-2026-03-17
1089,2,2261,indirect,0.396667,NaN,NaN,NaN,NaN,0.206667,gpt-5.4-mini-2026-03-17
1090,2,2262,indirect,0.916667,NaN,NaN,NaN,NaN,0.833333,gpt-5.4-mini-2026-03-17
1091,2,2263,indirect,0.396667,NaN,NaN,NaN,NaN,0.206667,gpt-5.4-mini-2026-03-17
1092,2,2264,indirect,0.396667,NaN,NaN,NaN,NaN,0.206667,gpt-5.4-mini-2026-03-17
1093,2,2265,indirect,0.916667,NaN,NaN,NaN,NaN,0.833333,gpt-5.4-mini-2026-03-17


### Visualization

In [19]:
fig = px.strip(df_long, y="FUN_mean", x="evaluator", color="evaluator", stripmode='group')
fig.update_traces(marker_size=3)
fig.show()

In [20]:
fig = px.strip(df_long, y="CER_mean", x="evaluator", color="evaluator", stripmode='group')
fig.update_traces(marker_size=3)
fig.show()